# 🏆 Ames Housing Price Prediction | Top 20% Solution
> **Pipeline Overview:** Advanced Feature Engineering, Skew Correction, 10-Fold Stratified Cross-Validation, and a 4-Model Weighted Super-Blend (CatBoost + LightGBM + XGBoost + Ridge).

---

## 1. Imports, Setup & Data Loading
In this step, we import core dependencies, ignore non-critical warnings, drop the 2 well-known extreme `GrLivArea` outliers from the training set, and merge data for joint feature transformation.

In [6]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd


import pandas.core.strings.accessor as string_accessor
pd.core.strings.StringMethods = string_accessor.StringMethods

from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


KAGGLE_PATH = '/kaggle/input/house-prices-advanced-regression-techniques'
LOCAL_PATH = 'Kaggle Houses Competition'  

if os.path.exists(KAGGLE_PATH):
    DATA_DIR = KAGGLE_PATH
    print(" Running on Kaggle Environment")
elif os.path.exists(LOCAL_PATH):
    DATA_DIR = LOCAL_PATH
    print(" Running on Local Environment")
else:
    
    DATA_DIR = '.' 
    print(" Using current working directory")

# 2. Loading data
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# Drop 2 main outliers
train = train.drop(train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index)

y_train_log = np.log1p(train['SalePrice'])
test_ids = test['Id']

df_all = pd.concat([train.drop(columns=['SalePrice', 'Id']), test.drop(columns=['Id'])], ignore_index=True)
print(f"Dataset successfully loaded! Shape: {df_all.shape}")

 Running on Local Environment
Dataset successfully loaded! Shape: (2917, 79)


## 2. Ordinal Encoding, Feature Engineering & Skew Correction
* **Ordinal Mapping:** Map structural quality metrics (`Po` -> `Ex`) into integer ranks.
* **Compound Domain Metrics:** Aggregated total space (`TotalSF`), total porch area (`TotalPorch`), and total bathrooms (`TotalBath`).
* **Interactions:** Interaction term `OverallQual_SF` to emphasize volatile high-value properties.
* **Skewness Mitigation:** Apply `np.log1p` to numerical features with skewness $> 0.75$.

In [3]:
# Ordinal Encoding
qual_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
             'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']

for col in qual_cols:
    df_all[col] = df_all[col].map(qual_map).fillna(0).astype(int)

# Domain-specific Aggregated Features
df_all['TotalSF'] = df_all['TotalBsmtSF'].fillna(0) + df_all['1stFlrSF'].fillna(0) + df_all['2ndFlrSF'].fillna(0)
df_all['TotalPorch'] = (df_all['WoodDeckSF'].fillna(0) + df_all['OpenPorchSF'].fillna(0) + 
                         df_all['EnclosedPorch'].fillna(0) + df_all['3SsnPorch'].fillna(0) + 
                         df_all['ScreenPorch'].fillna(0))
df_all['TotalBath'] = (df_all['FullBath'].fillna(0) + 0.5 * df_all['HalfBath'].fillna(0) + 
                       df_all['BsmtFullBath'].fillna(0) + 0.5 * df_all['BsmtHalfBath'].fillna(0))

df_all['HouseAge'] = df_all['YrSold'] - df_all['YearBuilt']
df_all['RemodAge'] = df_all['YrSold'] - df_all['YearRemodAdd']
df_all['IsNew'] = (df_all['YrSold'] == df_all['YearBuilt']).astype(int)
df_all['OverallQual_SF'] = df_all['TotalSF'] * df_all['OverallQual']

# Log Transform Highly Skewed Features
num_cols = df_all.select_dtypes(include=[np.number]).columns
skewed = df_all[num_cols].apply(lambda x: x.skew()).sort_values(ascending=False)
high_skew = skewed[skewed > 0.75].index

for col in high_skew:
    df_all[col] = np.log1p(df_all[col].clip(lower=0))

# One-Hot Encoding & Median Imputation
df_all = pd.get_dummies(df_all)
df_all = df_all.fillna(df_all.median())

# Split back to Train/Test
X = df_all.iloc[:len(train)].copy()
X_test = df_all.iloc[len(train):].copy()

# Standard Scaling for Linear Models (Ridge)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

print(f"Processed Features Count: {X.shape[1]}")

Processed Features Count: 261


## 3. 10-Fold Cross-Validation & Ensemble Blending
We train a 4-model ensemble across 10 folds:
- **CatBoost Regressor** (35%)
- **LightGBM Regressor** (25%)
- **XGBoost Regressor** (20%)
- **RidgeCV Linear Anchoring** (20%)

In [4]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

ridge_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))
lgb_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_train_log), 1):
    X_tr_s, y_tr = X_scaled[train_idx], y_train_log.iloc[train_idx]
    X_va_s, y_va = X_scaled[val_idx], y_train_log.iloc[val_idx]
    X_tr_b, X_va_b = X.iloc[train_idx], X.iloc[val_idx]
    
    # 1. Ridge Regression (Linear Anchor)
    m_ridge = RidgeCV(alphas=np.logspace(1, 3, 30))
    m_ridge.fit(X_tr_s, y_tr)
    
    # 2. CatBoost Regressor
    m_cat = CatBoostRegressor(iterations=1400, learning_rate=0.02, depth=6, random_seed=42, verbose=0)
    m_cat.fit(X_tr_b, y_tr, eval_set=(X_va_b, y_va), early_stopping_rounds=50)
    
    # 3. XGBoost Regressor
    m_xgb = XGBRegressor(n_estimators=1200, learning_rate=0.018, max_depth=4, subsample=0.7, colsample_bytree=0.7, random_state=42)
    m_xgb.fit(X_tr_b, y_tr, eval_set=[(X_va_b, y_va)], verbose=False)
    
    # 4. LightGBM Regressor
    m_lgb = LGBMRegressor(n_estimators=1200, learning_rate=0.018, max_depth=4, num_leaves=15, subsample=0.7, colsample_bytree=0.7, random_state=42, verbose=-1)
    m_lgb.fit(X_tr_b, y_tr, eval_set=[(X_va_b, y_va)])
    
    # Accumulate Test Predictions
    ridge_preds += m_ridge.predict(X_test_scaled) / 10
    cat_preds += m_cat.predict(X_test) / 10
    xgb_preds += m_xgb.predict(X_test) / 10
    lgb_preds += m_lgb.predict(X_test) / 10

# Final Super-Blend Ensembling
final_log = 0.35 * cat_preds + 0.25 * lgb_preds + 0.20 * xgb_preds + 0.20 * ridge_preds

# Boundary Clipping to Protect Against Extreme Outliers
min_log = y_train_log.min() - 0.05
max_log = y_train_log.max() + 0.05
final_log = np.clip(final_log, min_log, max_log)

print("Modeling & Blending Complete!")

Modeling & Blending Complete!


## 4. Export Submission
Revert log-transformed predictions back to actual USD scale using `np.expm1`.

In [5]:
submission = pd.DataFrame({
    'Id': test_ids.astype(int),
    'SalePrice': np.expm1(final_log)
})

submission.to_csv('submission.csv', index=False)
print("Submission generated successfully!")
submission.head()

Submission generated successfully!


,Id,SalePrice
0,1461,123339.072014
1,1462,163296.335874
2,1463,180444.315326
3,1464,196214.273348
4,1465,185670.466512
